In [0]:
df = spark.table("workspace.retail_project.bronze_sales")
display(df.limit(10))

In [0]:
df.printSchema()

In [0]:
from pyspark.sql import functions as F

null_counts = df.select([F.count(F.when(F.col(c).isNull(), 1)).alias(c)
for c in df.columns
])

display(null_counts)


In [0]:
duplicate_orders = (
    df.groupBy("ORDERNUMBER", "ORDERLINENUMBER")
    .count()
    .filter(F.col("count") > 1)
)

display(duplicate_orders)


In [0]:
df.select("ORDERDATE").show(10, truncate=False)

In [0]:
df_clean = df.withColumn(
    "ORDERDATE",
    F.to_date(F.col("ORDERDATE"), "M/d/yyyy H:mm"))

df_clean.select("ORDERDATE").show(10)

In [0]:
df_clean.select("ORDERDATE").printSchema()

In [0]:
invalid_dates =  df_clean.filter(F.col("ORDERDATE").isNull())
print("Invalid dates:", invalid_dates.count())

In [0]:

df_clean = (
    df_clean
    .withColumn("SALES", F.col("SALES").cast("decimal(18,2)"))
    .withColumn("PRICEEACH", F.col("PRICEEACH").cast("decimal(18,2)"))
    .withColumn("MSRP", F.col("MSRP").cast("decimal(18,2)"))
)


In [0]:

df_clean.select("SALES", "PRICEEACH", "MSRP").printSchema()


In [0]:

invalid_sales = df_clean.filter(
    F.col("SALES").isNull() |
    (F.col("SALES") < 0) |
    F.col("PRICEEACH").isNull() |
    (F.col("PRICEEACH") < 0) |
    F.col("QUANTITYORDERED").isNull() |
    (F.col("QUANTITYORDERED") <= 0)
)

print("Invalid sales records:", invalid_sales.count())


In [0]:

df_clean = df_clean.drop("ADDRESSLINE2")

print("Total columns:", len(df_clean.columns))


In [0]:

df_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.retail_project.silver_sales")


In [0]:

silver_df = spark.table("workspace.retail_project.silver_sales")

print("Silver rows:", silver_df.count())
print("Silver columns:", len(silver_df.columns))
